# CUSUM

The cumulative sum (CUSUM) control chart detects a sustained shift in the mean of a series. It adds up small deviations from the baseline, so a shift too small to notice in any single observation still triggers an alarm.

In [ ]:
using Plots, Statistics, Random

Random.seed!(3);

## Baseline and New Data

A clean baseline period, then new data whose mean shifts up by 1 standard deviation at time 100. The shift is hard to see by eye.

In [ ]:
x_baseline = 10 .+ 2 .* randn(200)
x_new = vcat(10 .+ 2 .* randn(100), 12 .+ 2 .* randn(100))

plot(x_new; xlabel = "Time", ylabel = "Value", label = "New data")
vline!([100]; color = :gray, linestyle = :dash, label = "True change")

## Two-Sided CUSUM

Standardise with the baseline mean and standard deviation. The upper statistic accumulates upward deviations beyond the allowance k, the lower one downward deviations. An alarm is raised when either exceeds h.

In [ ]:
function cusum(z, k)
    S_up, S_down = zeros(length(z)), zeros(length(z))
    for t in eachindex(z)
        S_up[t]   = max(0.0, (t == 1 ? 0.0 : S_up[t-1]) + z[t] - k)
        S_down[t] = max(0.0, (t == 1 ? 0.0 : S_down[t-1]) - z[t] - k)
    end
    return S_up, S_down
end

mu0, sigma0 = mean(x_baseline), std(x_baseline)
z = (x_new .- mu0) ./ sigma0
k, h = 0.5, 5.0
S_up, S_down = cusum(z, k)
alarms = findall((S_up .> h) .| (S_down .> h))
println("First alarm at: ", isempty(alarms) ? "none" : alarms[1])

In [ ]:
plot([S_up S_down]; label = ["Upper CUSUM" "Lower CUSUM"], xlabel = "Time", ylabel = "S(t)")
hline!([h]; color = :red, linestyle = :dash, label = "Threshold h")
vline!([100]; color = :gray, linestyle = :dash, label = "True change")

## When Did the Change Happen?

The alarm comes after the change. The last time the upper statistic was zero before the alarm estimates the change point.

In [ ]:
if !isempty(alarms)
    start = findlast(==(0.0), S_up[1:alarms[1]])
    println("Estimated change after time: ", something(start, 0))
end

## False Alarms: Average Run Length

On in-control data the statistic still crosses h eventually. The average run length (ARL0) is the mean time to a false alarm. A short simulation estimates it.

In [ ]:
function run_length(k, h; maxlen = 5000)
    S_up, S_down = 0.0, 0.0
    for t in 1:maxlen
        e = randn()
        S_up, S_down = max(0.0, S_up + e - k), max(0.0, S_down - e - k)
        (S_up > h || S_down > h) && return t
    end
    return maxlen
end

println("Estimated ARL0 (k = 0.5, h = 5): ", round(mean(run_length(0.5, 5.0) for _ in 1:300)))

## Tuning k and h

k is half the shift to detect, in standard deviations. Pairs with an ARL0 near 370 trade sensitivity to small and large shifts.

In [ ]:
for (kk, hh) in [(0.25, 8.0), (0.5, 4.77), (1.0, 2.5)]
    su, sd = cusum(z, kk)
    a = findfirst((su .> hh) .| (sd .> hh))
    println("k = ", rpad(kk, 5), "h = ", rpad(hh, 5), "first alarm: ", something(a, "none"))
end

## Summary
- CUSUM accumulates deviations from a baseline and detects sustained mean shifts.
- k is half the target shift; h sets the false-alarm rate (ARL0).
- The alarm lags the change; the last zero of the statistic estimates when it started.
- Use independent observations; for autocorrelated series, monitor model residuals.